In [24]:
# Step 1: Load the baseline dataset
import pandas as pd

df = pd.read_csv('alphatrade_phase1.csv')

print(df.shape)
df.head()

(1508, 19)


,Date,Close,High,Low,Open,Volume,SMA_20,SMA_50,EMA_20,EMA_50,Daily_Return,Volatility,RSI,MACD,Signal_Line,BB_Middle,BB_Upper,BB_Lower,Signal
0,2020-01-02,72.333855,72.394063,71.091161,71.344032,135480400,NaN,NaN,72.333855,72.333855,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0
1,2020-01-03,71.630630,72.389250,71.406659,71.563198,146322800,NaN,NaN,72.266881,72.306277,-0.009722,NaN,NaN,-0.056098,-0.011220,NaN,NaN,NaN,0
2,2020-01-06,72.201401,72.239935,70.503539,70.754006,118387200,NaN,NaN,72.260645,72.302164,0.007968,NaN,NaN,-0.053878,-0.019751,NaN,NaN,NaN,0
3,2020-01-07,71.861855,72.466338,71.642697,72.211056,108872000,NaN,NaN,72.222665,72.284897,-0.004703,NaN,NaN,-0.078611,-0.031523,NaN,NaN,NaN,0
4,2020-01-08,73.017815,73.318854,71.565599,71.565599,132079200,NaN,NaN,72.298393,72.313639,0.016086,NaN,NaN,-0.004880,-0.026195,NaN,NaN,NaN,0


In [25]:
'''Step 2: Create Lag Features
Concept
These tell the model what happened in previous days.'''

df['Lag_1'] = df['Close'].shift(1)
df['Lag_2'] = df['Close'].shift(2)
df['Lag_3'] = df['Close'].shift(3)
df['Lag_5'] = df['Close'].shift(5)

In [26]:
'''Step 3: Create Momentum Features
Concept
These measure how much the stock has moved.'''

df['Momentum_5'] = df['Close'] - df['Close'].shift(5)
df['Momentum_10'] = df['Close'] - df['Close'].shift(10)

In [27]:
'''Step 4: Create Rolling Statistics
Concept
These capture short-term trend and volatility.'''

df['Rolling_Mean_5'] = df['Close'].rolling(5).mean()

df['Rolling_STD_5'] = df['Close'].rolling(5).std()

In [28]:
'''Step 5: Create Percentage Returns
Concept
These measure percentage movement over time.'''

df['Return_5'] = df['Close'].pct_change(5)

df['Return_10'] = df['Close'].pct_change(10)

In [29]:
# Step 6: Remove the newly created NaN values
df = df.dropna()

In [30]:
# Step 7: Verify
print(df.shape)
print(df.isnull().sum().sum())
print(df.columns)

(1459, 29)
0
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal', 'Lag_1',
       'Lag_2', 'Lag_3', 'Lag_5', 'Momentum_5', 'Momentum_10',
       'Rolling_Mean_5', 'Rolling_STD_5', 'Return_5', 'Return_10'],
      dtype='str')


In [31]:
print(df.shape)
print(df.columns)

(1459, 29)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal', 'Lag_1',
       'Lag_2', 'Lag_3', 'Lag_5', 'Momentum_5', 'Momentum_10',
       'Rolling_Mean_5', 'Rolling_STD_5', 'Return_5', 'Return_10'],
      dtype='str')


In [32]:
'''Step 1: Create the new feature list
This time, include both the old and advanced features:'''
features = [
    # Original features
    'Close',
    'Volume',
    'SMA_20',
    'SMA_50',
    'EMA_20',
    'EMA_50',
    'Daily_Return',
    'Volatility',
    'RSI',
    'MACD',
    'Signal_Line',

    # Lag features
    'Lag_1',
    'Lag_2',
    'Lag_3',
    'Lag_5',

    # Momentum features
    'Momentum_5',
    'Momentum_10',

    # Rolling statistics
    'Rolling_Mean_5',
    'Rolling_STD_5',

    # Return features
    'Return_5',
    'Return_10'
]

In [33]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

df = df[:-1]

In [34]:
df = df.dropna()

In [35]:
print(df['Target'].value_counts())
print(df.shape)

Target
1    778
0    680
Name: count, dtype: int64
(1458, 30)


In [36]:
X = df[features]
y = df['Target']

print(X.shape)
print(y.shape)

(1458, 21)
(1458,)


In [37]:
# Chronological Split 
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(1166, 21)
(292, 21)
(1166,)
(292,)


In [38]:
# Then train Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.5


In [39]:
# Then train XGBoost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

XGBoost Accuracy: 0.5068493150684932


In [40]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False)

print(importance)

           Feature  Importance
6     Daily_Return    0.062757
18   Rolling_STD_5    0.060631
7       Volatility    0.060376
1           Volume    0.056197
8              RSI    0.055712
16     Momentum_10    0.054850
20       Return_10    0.052863
15      Momentum_5    0.050828
10     Signal_Line    0.050567
19        Return_5    0.049422
9             MACD    0.047523
14           Lag_5    0.043717
0            Close    0.041886
12           Lag_2    0.041740
5           EMA_50    0.041016
3           SMA_50    0.040795
13           Lag_3    0.040265
11           Lag_1    0.039890
4           EMA_20    0.038710
2           SMA_20    0.036031
17  Rolling_Mean_5    0.034226


In [41]:
# Feature Pruning:
best_features = [
    'Daily_Return',
    'Rolling_STD_5',
    'Volatility',
    'Volume',
    'RSI',
    'Momentum_10',
    'Return_10',
    'Momentum_5',
    'Signal_Line',
    'Return_5',
    'MACD'
]

In [42]:
X = df[best_features]
y = df['Target']

print(X.shape)
print(y.shape)

(1458, 11)
(1458,)


In [43]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [44]:
print(X_train.shape)
print(X_test.shape)

(1166, 11)
(292, 11)


In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.4897260273972603


In [46]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

XGBoost Accuracy: 0.4794520547945205
